In [58]:
import pandas as pd
import numpy as np
import os

dataset = pd.DataFrame(columns=["row","col","r","c","value","status"])
for file in os.listdir("dataset"):
    if file.endswith(".csv"):
        dataset_file = pd.read_csv(os.path.join("dataset", file))
        print(f"Processing {file}...")
        dataset = pd.concat([dataset, dataset_file], ignore_index=True)

# print(len(dataset))
print(dataset.head())

Processing idescat-elepc-6011-1.csv...
Processing idescat-elepc-6011-1_2015.csv...
Processing idescat-elepc-6011-1_2021.csv...
Processing idescat-elepc-6011-1_2024.csv...
Processing idescat-elepc-6011_2017.csv...
Processing idescat-elepc-votants.csv...
    row  col  r  c      value status
0  2012  CiU  1  1  1112605.0    NaN
1  2012  PSC  1  2   523537.0    NaN
2  2012   PP  1  3   470759.0    NaN
3  2012   IC  1  4   358860.0    NaN
4  2012  ERC  1  5   496466.0    NaN


In [59]:
dataset.rename(columns={"row": "year", "col": "party", "value": "votes"}, inplace=True)
dataset.drop(columns=["r", "c", "status"], inplace=True)

dataset['party'] = dataset['party'].replace(["JUNTSxCAT", "CiU", "CAT-JUNTS+", "JxCat"], "JUNTS")
dataset['party'] = dataset['party'].replace(["ERC-Cat Sí"], "ERC")
dataset["party"] = dataset["party"].replace(["COMUNS SUMAR", "Cat Comú-Podem", "ECP-PEC", "CatSíqueesPot", "IC"], "COMUNS")
dataset['party'] = dataset["party"].replace(["CUP-G", "CUP-DT"], "CUP")
dataset['party'] = dataset['party'].replace(["C's"], "Cs")

dataset["year"] = dataset["year"].astype(int)

dataset.head()

,year,party,votes
0,2012,JUNTS,1112605.0
1,2012,PSC,523537.0
2,2012,PP,470759.0
3,2012,COMUNS,358860.0
4,2012,ERC,496466.0


In [60]:
for col in ["year", "party"]:
    print(f"{col}: {dataset[col].nunique()} unique values")
    print(f"    {dataset[col].unique()}")

year: 14 unique values
    [2012 2010 2006 2003 1999 1995 1992 1988 1984 1980 2015 2021 2024 2017]
party: 19 unique values
    ['JUNTS' 'PSC' 'PP' 'COMUNS' 'ERC' 'Cs' 'CUP' 'Altres candidatures'
 'Total' 'JxSí' 'VOX' 'ALIANÇA.CAT' 'Nombre / electors' 'Nombre / votants'
 'Vots / a candidatures' 'Vots / blancs' 'Vots / nuls'
 'Percentatge / participació' 'Percentatge / abstenció']


In [61]:
df_cols = dataset["party"].unique()

# Sort cols by votes in the most recent year
latest_year = dataset["year"].max()
latest_year_data = dataset[dataset["year"] == latest_year]
df_cols = latest_year_data.sort_values("votes", ascending=False)["party"].unique()

df = pd.DataFrame(columns=df_cols)
for _, row in dataset.iterrows():
    year = row["year"]
    party = row["party"]
    votes = row["votes"]
    df.loc[year, party] = votes
df.sort_index(inplace=True)
print(df.shape)
df

(14, 19)


,Nombre / electors,Nombre / votants,Vots / a candidatures,Total,PSC,JUNTS,ERC,PP,VOX,COMUNS,CUP,ALIANÇA.CAT,Altres candidatures,Vots / blancs,Vots / nuls,Percentatge / participació,Percentatge / abstenció,Cs,JxSí
1980,4436463.0,2725558.0,2693986.0,2693986.0,608689.0,754448.0,241711.0,64119.0,NaN,509014.0,NaN,NaN,516005.0,18004.0,13568.0,61.4,38.6,NaN,NaN
1984,4494966.0,2891663.0,2861908.0,2862121.0,865449.0,1345513.0,126865.0,221697.0,NaN,160586.0,NaN,NaN,142011.0,14241.0,15514.0,64.3,35.7,NaN,NaN
1988,4550969.0,2703700.0,2673123.0,2673123.0,800999.0,1230356.0,111276.0,143062.0,NaN,208689.0,NaN,NaN,178741.0,16922.0,13655.0,59.4,40.6,NaN,NaN
1992,4816375.0,2648172.0,2605981.0,2605981.0,726099.0,1218831.0,209881.0,157395.0,NaN,171455.0,NaN,NaN,122320.0,31075.0,11116.0,55.0,45.0,NaN,NaN
1995,5029717.0,3218522.0,3178235.0,3178235.0,797422.0,1314548.0,304833.0,420341.0,NaN,312371.0,NaN,NaN,28720.0,31393.0,8894.0,64.0,36.0,NaN,NaN
1999,5206419.0,3118410.0,3081736.0,3081736.0,1177777.0,1172721.0,270176.0,295765.0,NaN,78213.0,NaN,NaN,87084.0,28935.0,7739.0,59.9,40.1,NaN,NaN
2003,5207420.0,3300637.0,3261741.0,3261741.0,1026396.0,1018164.0,542046.0,390882.0,NaN,240235.0,NaN,NaN,44018.0,30160.0,8736.0,63.4,36.6,NaN,NaN
2006,5212423.0,2959450.0,2885893.0,2885893.0,789956.0,928936.0,414044.0,313368.0,NaN,281405.0,NaN,NaN,68640.0,60089.0,13468.0,56.8,43.2,89544.0,NaN
2010,5229873.0,3135195.0,3021460.0,3021460.0,570405.0,1198193.0,218152.0,384470.0,NaN,229853.0,NaN,NaN,314503.0,91475.0,22260.0,60.0,40.1,105884.0,NaN
2012,5257351.0,3657753.0,3572806.0,3572806.0,523537.0,1112605.0,496466.0,470759.0,NaN,358860.0,126198.0,NaN,209729.0,52826.0,32121.0,69.6,30.4,274652.0,NaN


In [62]:
# Redistribute 2015 JxSí votes between ERC and JUNTS
# using the average coalition proportions from 2012 and 2017

coalition_year = 2015
reference_years = [2012, 2017]
parties = ["ERC", "JUNTS"]

if coalition_year in df.index:

    # Total coalition votes in 2015
    jxsi_votes = df.at[coalition_year, "JxSí"]

    # Compute party proportions in each reference year
    proportions = (
        df.loc[reference_years, parties]
        .div(df.loc[reference_years, parties].sum(axis=1), axis=0)
    )

    # Average proportions across reference years
    avg_proportions = proportions.mean()

    # Redistribute coalition votes
    redistributed_votes = (jxsi_votes * avg_proportions).round().astype(int)

    # Fill missing values
    df.loc[coalition_year, parties] = redistributed_votes.values

    # Remove coalition column afterward
    df = df.drop(columns="JxSí")

    print("Average proportions:")
    print(avg_proportions.round(4))

    print("\nRedistributed votes:")
    print(redistributed_votes)
    
print(df.shape)
df


Average proportions:
ERC      0.4026
JUNTS    0.5974
dtype: object

Redistributed votes:
ERC      655768
JUNTS    972946
dtype: int64
(14, 18)


,Nombre / electors,Nombre / votants,Vots / a candidatures,Total,PSC,JUNTS,ERC,PP,VOX,COMUNS,CUP,ALIANÇA.CAT,Altres candidatures,Vots / blancs,Vots / nuls,Percentatge / participació,Percentatge / abstenció,Cs
1980,4436463.0,2725558.0,2693986.0,2693986.0,608689.0,754448.0,241711.0,64119.0,NaN,509014.0,NaN,NaN,516005.0,18004.0,13568.0,61.4,38.6,NaN
1984,4494966.0,2891663.0,2861908.0,2862121.0,865449.0,1345513.0,126865.0,221697.0,NaN,160586.0,NaN,NaN,142011.0,14241.0,15514.0,64.3,35.7,NaN
1988,4550969.0,2703700.0,2673123.0,2673123.0,800999.0,1230356.0,111276.0,143062.0,NaN,208689.0,NaN,NaN,178741.0,16922.0,13655.0,59.4,40.6,NaN
1992,4816375.0,2648172.0,2605981.0,2605981.0,726099.0,1218831.0,209881.0,157395.0,NaN,171455.0,NaN,NaN,122320.0,31075.0,11116.0,55.0,45.0,NaN
1995,5029717.0,3218522.0,3178235.0,3178235.0,797422.0,1314548.0,304833.0,420341.0,NaN,312371.0,NaN,NaN,28720.0,31393.0,8894.0,64.0,36.0,NaN
1999,5206419.0,3118410.0,3081736.0,3081736.0,1177777.0,1172721.0,270176.0,295765.0,NaN,78213.0,NaN,NaN,87084.0,28935.0,7739.0,59.9,40.1,NaN
2003,5207420.0,3300637.0,3261741.0,3261741.0,1026396.0,1018164.0,542046.0,390882.0,NaN,240235.0,NaN,NaN,44018.0,30160.0,8736.0,63.4,36.6,NaN
2006,5212423.0,2959450.0,2885893.0,2885893.0,789956.0,928936.0,414044.0,313368.0,NaN,281405.0,NaN,NaN,68640.0,60089.0,13468.0,56.8,43.2,89544.0
2010,5229873.0,3135195.0,3021460.0,3021460.0,570405.0,1198193.0,218152.0,384470.0,NaN,229853.0,NaN,NaN,314503.0,91475.0,22260.0,60.0,40.1,105884.0
2012,5257351.0,3657753.0,3572806.0,3572806.0,523537.0,1112605.0,496466.0,470759.0,NaN,358860.0,126198.0,NaN,209729.0,52826.0,32121.0,69.6,30.4,274652.0


In [63]:
df["Abstenció"] = df["Nombre / electors"] - df["Nombre / votants"] 
df = df.drop(columns=['Nombre / electors', 'Nombre / votants',
                  'Vots / a candidatures', 'Vots / blancs', 'Vots / nuls',
                  'Percentatge / participació', 'Percentatge / abstenció'])
print(df.columns)

Index(['Total', 'PSC', 'JUNTS', 'ERC', 'PP', 'VOX', 'COMUNS', 'CUP',
       'ALIANÇA.CAT', 'Altres candidatures', 'Cs', 'Abstenció'],
      dtype='str')


In [64]:
df.replace(np.nan, 0, inplace=True)
df.drop(columns=["Total"], inplace=True, errors="ignore")

# Make all columns int
for col in df.columns:
    df[col] = df[col].astype(int)

df

,PSC,JUNTS,ERC,PP,VOX,COMUNS,CUP,ALIANÇA.CAT,Altres candidatures,Cs,Abstenció
1980,608689,754448,241711,64119,0,509014,0,0,516005,0,1710905
1984,865449,1345513,126865,221697,0,160586,0,0,142011,0,1603303
1988,800999,1230356,111276,143062,0,208689,0,0,178741,0,1847269
1992,726099,1218831,209881,157395,0,171455,0,0,122320,0,2168203
1995,797422,1314548,304833,420341,0,312371,0,0,28720,0,1811195
1999,1177777,1172721,270176,295765,0,78213,0,0,87084,0,2088009
2003,1026396,1018164,542046,390882,0,240235,0,0,44018,0,1906783
2006,789956,928936,414044,313368,0,281405,0,0,68640,89544,2252973
2010,570405,1198193,218152,384470,0,229853,0,0,314503,105884,2094678
2012,523537,1112605,496466,470759,0,358860,126198,0,209729,274652,1599598


In [65]:
# Normalize by total votes per year
df = df.div(df.sum(axis=1), axis=0)
print(df.shape)
df

(14, 11)


,PSC,JUNTS,ERC,PP,VOX,COMUNS,CUP,ALIANÇA.CAT,Altres candidatures,Cs,Abstenció
1980,0.138185,0.171275,0.054873,0.014556,0.000000,0.115557,0.000000,0.000000,0.117144,0.000000,0.388410
1984,0.193811,0.301318,0.028411,0.049647,0.000000,0.035962,0.000000,0.000000,0.031802,0.000000,0.359048
1988,0.177197,0.272179,0.024616,0.031648,0.000000,0.046166,0.000000,0.000000,0.039541,0.000000,0.408652
1992,0.152089,0.255296,0.043962,0.032968,0.000000,0.035913,0.000000,0.000000,0.025621,0.000000,0.454152
1995,0.159822,0.263467,0.061096,0.084246,0.000000,0.062607,0.000000,0.000000,0.005756,0.000000,0.363006
1999,0.227821,0.226843,0.052261,0.057211,0.000000,0.015129,0.000000,0.000000,0.016845,0.000000,0.403890
2003,0.198586,0.196993,0.104874,0.075627,0.000000,0.046480,0.000000,0.000000,0.008517,0.000000,0.368922
2006,0.153722,0.180767,0.080571,0.060980,0.000000,0.054760,0.000000,0.000000,0.013357,0.017425,0.438418
2010,0.111491,0.234199,0.042640,0.075148,0.000000,0.044927,0.000000,0.000000,0.061473,0.020696,0.409426
2012,0.101217,0.215104,0.095984,0.091014,0.000000,0.069380,0.024398,0.000000,0.040548,0.053099,0.309256


In [66]:
# floats to 6 decimals
for col in df.columns:
    df[col] = df[col].round(6)

for row in df.index:
    row_sum = df.loc[row].sum()
    if not np.isclose(row_sum, 1.0):
        print(f"Warning: Row {row} sums to {row_sum:.6f} instead of 1.0")

df.to_csv("data.csv", index=True)
print("Data processing complete. Cleaned data saved to data.csv")

Data processing complete. Cleaned data saved to data.csv
